# Prepare activity history for recommendation ML

The notebook creates two files without modifying the original history:

- `activity_history_enriched.csv`: all history rows with joined employee/event metadata and causal historical features.
- `activity_completion_logistic.csv`: voluntary activities with finalized outcomes, ready for a Logistic Regression pipeline.

All historical aggregates for a row use only dates strictly earlier than that row's date. Rows from the same date are evaluated before that date is added to the running statistics. Outcome-time fields such as `completion_pct`, `score`, `feedback_rating`, and `status` are excluded from the Logistic Regression feature table.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path.cwd()
if not (DATA_DIR / "activity_history.csv").exists():
    candidate = DATA_DIR / "case_1" / "career_quest_dataset"
    if (candidate / "activity_history.csv").exists():
        DATA_DIR = candidate
    else:
        raise FileNotFoundError("Run the notebook from the dataset folder or the project root.")

HISTORY_PATH = DATA_DIR / "activity_history.csv"
EMPLOYEES_PATH = DATA_DIR / "employees_updated.json"
EVENTS_PATH = DATA_DIR / "events.json"
ENRICHED_PATH = DATA_DIR / "activity_history_enriched.csv"
LOGISTIC_PATH = DATA_DIR / "activity_completion_logistic.csv"
VALIDATION_START = pd.Timestamp("2026-05-01")

print(f"Dataset folder: {DATA_DIR.resolve()}")

In [ ]:
history = pd.read_csv(HISTORY_PATH)
history["date"] = pd.to_datetime(history["date"], errors="raise")
history["due_date"] = pd.to_datetime(history["due_date"], errors="coerce")

with EMPLOYEES_PATH.open(encoding="utf-8") as file:
    employees_document = json.load(file)
with EVENTS_PATH.open(encoding="utf-8") as file:
    events_document = json.load(file)

employees_by_id = {employee["employee_id"]: employee for employee in employees_document["employees"]}
events_by_id = {event["event_id"]: event for event in events_document["events"]}

assert history["record_id"].is_unique, "record_id must be unique"
assert set(history["employee_id"]) <= employees_by_id.keys(), "Unknown employee_id found"
assert set(history["event_id"]) <= events_by_id.keys(), "Unknown event_id found"

history = history.sort_values(["date", "employee_id", "event_id", "record_id"]).reset_index(drop=True)
print(f"Loaded {len(history)} history rows.")

In [ ]:
enriched = history.copy()

enriched["employee_role"] = enriched["employee_id"].map(lambda value: employees_by_id[value]["role"])
enriched["employee_grade"] = enriched["employee_id"].map(lambda value: employees_by_id[value]["grade"])
enriched["employee_work_format"] = enriched["employee_id"].map(lambda value: employees_by_id[value]["work_format"])
enriched["employee_preferred_language"] = enriched["employee_id"].map(lambda value: employees_by_id[value]["preferred_language"])
enriched["employee_hire_date"] = pd.to_datetime(
    enriched["employee_id"].map(lambda value: employees_by_id[value]["hire_date"])
)
enriched["event_type"] = enriched["event_id"].map(lambda value: events_by_id[value]["type"])
enriched["event_format"] = enriched["event_id"].map(lambda value: events_by_id[value]["format"])
enriched["duration_hours"] = enriched["event_id"].map(lambda value: events_by_id[value]["duration_hours"])
enriched["mandatory"] = enriched["event_id"].map(lambda value: events_by_id[value]["mandatory"])
enriched["tenure_days_at_activity"] = (enriched["date"] - enriched["employee_hire_date"]).dt.days

assert (enriched["tenure_days_at_activity"] >= 0).all(), "Activity before hire date detected"

In [ ]:
FINAL_VOLUNTARY_STATUSES = {"completed", "dropped", "no_show", "declined"}
OUTCOME_CATEGORIES = ("completed", "dropped", "no_show", "declined")

employee_outcomes = defaultdict(Counter)
employee_format_outcomes = defaultdict(Counter)
employee_type_outcomes = defaultdict(Counter)
event_outcomes = defaultdict(Counter)
employee_event_attempts = Counter()
last_employee_activity_date = {}

feature_rows = {}

def four_class_rates(counter):
    total = sum(counter[category] for category in OUTCOME_CATEGORIES)
    denominator = total + len(OUTCOME_CATEGORIES)
    return total, {category: (counter[category] + 1) / denominator for category in OUTCOME_CATEGORIES}

def completion_rate(counter):
    total = sum(counter[category] for category in OUTCOME_CATEGORIES)
    return total, (counter["completed"] + 1) / (total + 2)

for activity_date, daily_rows in enriched.groupby("date", sort=True):
    # Calculate every row of the day from state available before this day.
    for row_index, row in daily_rows.iterrows():
        employee_id = row["employee_id"]
        event_id = row["event_id"]
        event_format = row["event_format"]
        event_type = row["event_type"]

        employee_count, employee_rates = four_class_rates(employee_outcomes[employee_id])
        format_count, format_completion_rate = completion_rate(employee_format_outcomes[(employee_id, event_format)])
        type_count, type_completion_rate = completion_rate(employee_type_outcomes[(employee_id, event_type)])
        event_count, event_completion_rate = completion_rate(event_outcomes[event_id])
        previous_date = last_employee_activity_date.get(employee_id)

        feature_rows[row_index] = {
            "attempt_number": employee_event_attempts[(employee_id, event_id)] + 1,
            "prior_same_event_attempts": employee_event_attempts[(employee_id, event_id)],
            "employee_prior_outcomes": employee_count,
            "employee_completion_rate_before": employee_rates["completed"],
            "employee_dropped_rate_before": employee_rates["dropped"],
            "employee_no_show_rate_before": employee_rates["no_show"],
            "employee_declined_rate_before": employee_rates["declined"],
            "employee_format_prior_outcomes": format_count,
            "employee_format_completion_rate_before": format_completion_rate,
            "employee_type_prior_outcomes": type_count,
            "employee_type_completion_rate_before": type_completion_rate,
            "event_prior_outcomes": event_count,
            "event_completion_rate_before": event_completion_rate,
            "is_first_employee_activity": int(previous_date is None),
            "days_since_previous_activity": -1 if previous_date is None else (activity_date - previous_date).days,
        }

    # Only after feature calculation, update state with finalized voluntary outcomes.
    for _, row in daily_rows.iterrows():
        employee_id = row["employee_id"]
        event_id = row["event_id"]
        employee_event_attempts[(employee_id, event_id)] += 1
        last_employee_activity_date[employee_id] = activity_date

        if not row["mandatory"] and row["status"] in FINAL_VOLUNTARY_STATUSES:
            outcome = row["status"]
            employee_outcomes[employee_id][outcome] += 1
            employee_format_outcomes[(employee_id, row["event_format"])][outcome] += 1
            employee_type_outcomes[(employee_id, row["event_type"])][outcome] += 1
            event_outcomes[event_id][outcome] += 1

causal_features = pd.DataFrame.from_dict(feature_rows, orient="index").sort_index()
enriched = pd.concat([enriched, causal_features], axis=1)

enriched["outcome_class"] = "not_training_eligible"
voluntary_final = (~enriched["mandatory"]) & enriched["status"].isin(FINAL_VOLUNTARY_STATUSES)
enriched.loc[voluntary_final, "outcome_class"] = enriched.loc[voluntary_final, "status"]
enriched["completed_label"] = pd.NA
enriched.loc[voluntary_final, "completed_label"] = (
    enriched.loc[voluntary_final, "status"] == "completed"
).astype(int)

In [ ]:
logistic_columns = [
    "record_id", "employee_id", "event_id", "date",
    "completed_label",
    "employee_role", "employee_grade", "employee_work_format",
    "employee_preferred_language", "event_type", "event_format", "assigned_by",
    "duration_hours", "tenure_days_at_activity",
    "attempt_number", "prior_same_event_attempts",
    "employee_prior_outcomes", "employee_completion_rate_before",
    "employee_dropped_rate_before", "employee_no_show_rate_before",
    "employee_declined_rate_before",
    "employee_format_prior_outcomes", "employee_format_completion_rate_before",
    "employee_type_prior_outcomes", "employee_type_completion_rate_before",
    "event_prior_outcomes", "event_completion_rate_before",
    "is_first_employee_activity", "days_since_previous_activity",
]

logistic = enriched.loc[voluntary_final, logistic_columns].copy()
logistic["completed_label"] = logistic["completed_label"].astype(int)
logistic.insert(4, "dataset_split", "train")
logistic.loc[logistic["date"] >= VALIDATION_START, "dataset_split"] = "validation"

for rate_column in [column for column in logistic.columns if column.endswith("_rate_before")]:
    assert logistic[rate_column].between(0, 1).all(), f"Invalid rate: {rate_column}"

assert len(enriched) == len(history) == 2743
assert len(logistic) == 1503
assert logistic["completed_label"].value_counts().to_dict() == {1: 1044, 0: 459}
assert logistic.isna().sum().sum() == 0, "Logistic dataset contains missing values"
assert not {"status", "completion_pct", "score", "feedback_rating", "due_date"} & set(logistic.columns)

enriched.to_csv(ENRICHED_PATH, index=False, date_format="%Y-%m-%d")
logistic.to_csv(LOGISTIC_PATH, index=False, date_format="%Y-%m-%d")

print(f"Saved enriched history: {ENRICHED_PATH.resolve()} ({len(enriched)} rows)")
print(f"Saved logistic dataset: {LOGISTIC_PATH.resolve()} ({len(logistic)} rows)")
print(logistic.groupby(["dataset_split", "completed_label"]).size().rename("rows"))

In [ ]:
logistic.head(10)